# 03 · Preprocessing cache and training (Kaggle, 2× T4)

> Runs on Kaggle: the DICOM data (about 4,400 studies) stays there. Attach the competition data, enable the GPU
> and internet, and clone this repository. The outputs used for the results in notebook 05 are in `outputs/runs/`.

**Preprocessing** (`kneemri.preprocessing`): DICOM geometry → canonical orientation per plane, left knees mirrored to
right, crop to a physical field of view, percentile intensity clip, resample to a fixed slice count, uint8 cache.

**Model** (`kneemri.model.KneeNet`): each 2.5D window (3 adjacent slices) of each series → shared ConvNeXt-T →
slot and depth embeddings → a transformer layer across all windows of all series → one attention-pooling head per finding.

**Final ensemble:** three preprocessing views × 5 folds = 15 ConvNeXt-T models.

| Family | Field of view | Slices | Size | Augmentation |
|---|---|---|---|---|
| `cnxt_fov140` | 140 mm | 24 | 320 px | affine + gamma |
| `cnxt_zoom110` | 110 mm | 32 | 320 px | affine + gamma |
| `cnxt_native_mirror` | full native | 32 | 256 px | affine + gamma + anatomical mirror |

In [ ]:
!git clone -q https://github.com/lucky07-07/knee-mri-abnormality-detection.git /kaggle/working/repo
!pip install -q -e /kaggle/working/repo
COMP = "/kaggle/input/rsna-knee-abnormality-detection"

## 1. Build the cache (CPU notebook, once per family)
Large caches are split across notebooks with `--part` / `--n-parts` (Kaggle keeps about 500 output files and 20 GB per notebook).

In [ ]:
!python -m kneemri.cache --config /kaggle/working/repo/configs/native_mirror.yaml \
    --comp {COMP} --out /kaggle/working --part 0 --n-parts 3

## 2. Report labels
Run the LLM grading pass (GPU) and the blend from notebook 02, or attach the saved `blended_labels.csv` as a dataset.

In [ ]:
from kneemri.labels.llm import ReportGrader
import pandas as pd
train = pd.read_csv(f"{COMP}/train.csv")
grades = ReportGrader("Qwen/Qwen2.5-14B-Instruct").grade(train.Report.fillna(""))
grades.insert(0, "StudyInstanceUID", train.StudyInstanceUID.values)
grades.to_csv("/kaggle/working/report_llm_grades.csv", index=False)

## 3. Train 5 folds, one process per GPU

In [ ]:
import subprocess, threading, torch
CACHE = ["/kaggle/input/native-cache-p0", "/kaggle/input/native-cache-p1", "/kaggle/input/native-cache-p2"]
def run_folds(gpu, folds):
    for k in folds:
        subprocess.run(["python", "-u", "-m", "kneemri.train", "--config", "/kaggle/working/repo/configs/native_mirror.yaml",
                        "--fold", str(k), "--train-csv", f"{COMP}/train.csv",
                        "--labels", "/kaggle/input/knee-labels/blended_labels.csv", "--cache", *CACHE,
                        "--out", "/kaggle/working/runs"], env={**__import__("os").environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
n = max(1, torch.cuda.device_count())
threads = [threading.Thread(target=run_folds, args=(g, list(range(5))[g::n])) for g in range(n)]
[t.start() for t in threads]; [t.join() for t in threads]